# Notebook 3 - Segmentacion del tumor con U-Net (BUSI)

**Deep Learning en Python para el Modelado Predictivo** - Trabajo academico

Entrenamos una **U-Net** (Tema 5) para **segmentar el tumor pixel a pixel** usando el
**MISMO split** que el notebook 2 de clasificacion, y comparamos sus metricas IoU/DICE
con las del **Grad-CAM** del clasificador.

**Indice**
1. Carga de datos y split (el mismo que en clasificacion)
2. Analisis exploratorio
3. Arquitectura U-Net
4. Funcion de perdida y metricas (DICE)
5. Entrenamiento (con EarlyStopping)
6. Evaluacion en test (IoU / DICE)
7. Comparacion U-Net vs Grad-CAM del clasificador
8. Conclusiones

In [ ]:
# Montaje de Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, re, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from tensorflow.keras import Model, Input
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Conv2DTranspose, concatenate
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# ----------------- CONFIGURACION (AJUSTA si tu carpeta esta en otra ruta) -----------------
BASE     = '/content/drive/MyDrive/Deep Learning'
DATA_DIR = BASE + '/Dataset_BUSI_clean'   # la misma carpeta limpia del notebook 1
OUT_DIR  = BASE + '/resultados'           # donde el notebook 2 dejo split_indices.npz y localizacion.csv

CLASSES   = ['benign', 'malignant', 'normal']
IMG_SIZE  = (224, 224)
BATCH_SIZE = 16        # lote pequeno para que la U-Net entre en memoria
SEED = 42

MODO_PRUEBA = False
EPOCHS = 5 if MODO_PRUEBA else 60

os.makedirs(OUT_DIR, exist_ok=True)
np.random.seed(SEED); tf.random.set_seed(SEED)
assert os.path.isdir(DATA_DIR), 'No se encuentra DATA_DIR: ' + DATA_DIR
assert os.path.exists(os.path.join(OUT_DIR, 'split_indices.npz')), \
    'No se encuentra split_indices.npz -> ejecuta primero el notebook 2'
print('DATA_DIR =', DATA_DIR)

## 1. Carga de datos y split

Reutilizamos exactamente el mismo orden de carga que el notebook 2 (mismas imagenes
ordenadas igual) y cargamos los indices `idx_train / idx_test` desde el fichero que
dejo el notebook 2. **Esto garantiza el mismo conjunto de test.**

El notebook 2 ya no reserva un conjunto de validacion fijo. La U-Net si necesita uno
para el EarlyStopping, asi que aqui **extraemos una validacion interna estratificada
(15 % del train)**; el conjunto de **test (20 %) es identico** al de clasificacion, de
modo que la comparacion U-Net vs Grad-CAM sigue siendo justa.

Para la segmentacion **solo usamos las imagenes con tumor** (benigno y maligno) de cada
parte del split — las normales tienen mascara vacia y no aportan nada a la segmentacion.

In [ ]:
def cargar_dataset(data_dir):
    X, y, mascaras = [], [], []
    for idx, cls in enumerate(CLASSES):
        cdir = os.path.join(data_dir, cls)
        ficheros = sorted(os.listdir(cdir))
        imagenes = [f for f in ficheros if f.lower().endswith('.png') and '_mask' not in f.lower()]
        for f in imagenes:
            img = Image.open(os.path.join(cdir, f)).convert('RGB').resize(IMG_SIZE)
            X.append(np.array(img, dtype='float32'))
            y.append(idx)
            stem = f[:-4]
            mascaras.append([os.path.join(cdir, m) for m in ficheros if m.startswith(stem + '_mask')])
    return np.array(X), np.array(y), mascaras

t0 = time.time()
X, y, mascaras = cargar_dataset(DATA_DIR)
print('[TIEMPO] Carga de imagenes: %.1f s' % (time.time() - t0))
print('Imagenes totales:', X.shape)

split = np.load(os.path.join(OUT_DIR, 'split_indices.npz'))
idx_train, idx_test = split['idx_train'], split['idx_test']
print('Split del notebook 2 -> train:', len(idx_train), '| test:', len(idx_test))

In [ ]:
def cargar_mascara(lista):
    m = np.zeros(IMG_SIZE, np.uint8)
    for mp in lista:
        mm = np.array(Image.open(mp).convert('L').resize(IMG_SIZE, Image.NEAREST))
        m = np.maximum(m, (mm > 127).astype(np.uint8))
    return m

def construir_xy(indices):
    """Imagenes CON tumor (benigno/maligno) y su mascara. Devuelve tambien la clase
    de cada imagen para poder estratificar la validacion interna."""
    Xs, Ys, clases = [], [], []
    for i in indices:
        if CLASSES[y[i]] == 'normal':       # excluimos normales
            continue
        mask = cargar_mascara(mascaras[i])
        if mask.sum() == 0:                  # excluimos mascaras vacias
            continue
        Xs.append(X[i] / 255.0)               # normalizamos a [0,1]
        Ys.append(mask[..., np.newaxis].astype('float32'))
        clases.append(y[i])
    return (np.array(Xs, dtype='float32'),
            np.array(Ys, dtype='float32'),
            np.array(clases))

# Train completo (80%) y test (20%, identico al de clasificacion)
Xtr_full, Ytr_full, ctr_full = construir_xy(idx_train)
Xte, Yte, _                  = construir_xy(idx_test)

# Validacion interna para el EarlyStopping: 15% del train, estratificada por clase
i_tr, i_va = train_test_split(np.arange(len(Xtr_full)), test_size=0.15,
                              stratify=ctr_full, random_state=SEED)
Xtr, Ytr = Xtr_full[i_tr], Ytr_full[i_tr]
Xva, Yva = Xtr_full[i_va], Ytr_full[i_va]

print('Segmentacion -> train:', len(Xtr), '| val (interna):', len(Xva),
      '| test:', len(Xte))
print('Shape X train:', Xtr.shape, '| Shape Y train:', Ytr.shape)

## 2. Analisis exploratorio

In [ ]:
# Mostramos algunas imagenes con su mascara
plt.figure(figsize=(13, 6))
for k in range(4):
    plt.subplot(2, 4, k + 1)
    plt.imshow(Xtr[k]); plt.axis('off'); plt.title('imagen')
    plt.subplot(2, 4, k + 5)
    plt.imshow(Xtr[k]); plt.imshow(Ytr[k, ..., 0], alpha=0.45, cmap='Reds')
    plt.axis('off'); plt.title('+ mascara')
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'unet_muestras.png'), dpi=120); plt.show()

## 3. Arquitectura U-Net

Implementacion clasica (Ronneberger et al., 2015 — Tema 5):
- **Encoder**: 4 bloques de 2 convoluciones 3x3 + max pooling -> reduce de 224 a 14.
- **Bottleneck**: 2 convoluciones a 512 filtros.
- **Decoder**: 4 bloques con `Conv2DTranspose` ("convolucion transpuesta" del Tema 5,
  el upsampling aprendible) + **skip connections** (concatenacion con la capa
  correspondiente del encoder, contribucion clave de la U-Net).
- **Salida**: `Conv2D(1, 1)` con `sigmoid` -> mascara binaria 224x224.

In [ ]:
def bloque_conv(x, filtros):
    x = Conv2D(filtros, 3, padding='same', activation='relu')(x)
    x = Conv2D(filtros, 3, padding='same', activation='relu')(x)
    return x

def build_unet(input_shape=(224, 224, 3)):
    inputs = Input(input_shape)
    # Encoder
    c1 = bloque_conv(inputs, 32);  p1 = MaxPooling2D()(c1)
    c2 = bloque_conv(p1, 64);      p2 = MaxPooling2D()(c2)
    c3 = bloque_conv(p2, 128);     p3 = MaxPooling2D()(c3)
    c4 = bloque_conv(p3, 256);     p4 = MaxPooling2D()(c4)
    # Bottleneck
    bn = bloque_conv(p4, 512)
    # Decoder con skip connections
    u4 = Conv2DTranspose(256, 2, strides=2, padding='same')(bn)
    u4 = concatenate([u4, c4]);    c5 = bloque_conv(u4, 256)
    u3 = Conv2DTranspose(128, 2, strides=2, padding='same')(c5)
    u3 = concatenate([u3, c3]);    c6 = bloque_conv(u3, 128)
    u2 = Conv2DTranspose(64, 2, strides=2, padding='same')(c6)
    u2 = concatenate([u2, c2]);    c7 = bloque_conv(u2, 64)
    u1 = Conv2DTranspose(32, 2, strides=2, padding='same')(c7)
    u1 = concatenate([u1, c1]);    c8 = bloque_conv(u1, 32)
    salida = Conv2D(1, 1, activation='sigmoid')(c8)
    return Model(inputs, salida, name='U-Net')

unet = build_unet()
unet.summary()

## 4. Funcion de perdida y metricas: DICE

Como recomienda el Tema 5, usamos **Loss DICE = 1 - DICE** como funcion de perdida.
El **coeficiente DICE** mide el solapamiento entre la mascara predicha y la real:

DICE = 2 |A intersect B| / (|A| + |B|)

Y el **IoU** (intersection over union) mide la misma idea con una formula ligeramente
distinta:

IoU = |A intersect B| / |A union B|

Los dos valen entre 0 (sin solapamiento) y 1 (perfecto).

In [ ]:
def dice_coef(y_true, y_pred, smooth=1.0):
    yt = tf.reshape(y_true, [-1])
    yp = tf.reshape(y_pred, [-1])
    inter = tf.reduce_sum(yt * yp)
    return (2.0 * inter + smooth) / (tf.reduce_sum(yt) + tf.reduce_sum(yp) + smooth)

def dice_loss(y_true, y_pred):
    return 1.0 - dice_coef(y_true, y_pred)

## 5. Entrenamiento

Optimizador Adam con `lr=1e-4`, **EarlyStopping** monitorizando `val_loss`
(`patience=10`, `restore_best_weights=True`). El val decide cuando parar.

In [ ]:
# Si modelo_unet.keras ya existe en resultados/, se CARGA del disco en vez de
# reentrenar. Asi las metricas finales (IoU/DICE en test) son siempre las mismas
# entre ejecuciones, y si Colab se reinicia no pierdes el entrenamiento previo.
# Para forzar el reentrenamiento, borra modelo_unet.keras de la carpeta resultados/.
ruta_modelo = os.path.join(OUT_DIR, 'modelo_unet.keras')

if os.path.exists(ruta_modelo):
    print('U-Net YA ENTRENADA -> cargando', os.path.basename(ruta_modelo),
          '(no se re-entrena)')
    unet = tf.keras.models.load_model(
        ruta_modelo,
        custom_objects={'dice_loss': dice_loss, 'dice_coef': dice_coef})
    H = None
else:
    unet = build_unet()
    unet.compile(optimizer=Adam(1e-4), loss=dice_loss, metrics=[dice_coef])
    early = EarlyStopping(monitor='val_loss', patience=10,
                          restore_best_weights=True, verbose=1)
    t0 = time.time()
    H = unet.fit(Xtr, Ytr,
                 validation_data=(Xva, Yva),
                 epochs=EPOCHS, batch_size=BATCH_SIZE,
                 callbacks=[early], verbose=1)
    print('[TIEMPO] Entrenamiento U-Net: %.1f min  (%d epocas)' %
          ((time.time() - t0) / 60, len(H.history['loss'])))
    unet.save(ruta_modelo)
    print('Modelo U-Net guardado en', ruta_modelo)

In [ ]:
# Curvas de entrenamiento.
# Si la U-Net se ha cargado del disco (H is None) no hay historial que graficar.
if H is not None:
    plt.style.use('ggplot')
    plt.figure(figsize=(13, 4))
    ep = np.arange(len(H.history['loss']))
    plt.subplot(1, 2, 1)
    plt.plot(ep, H.history['loss'], label='train_loss (1-DICE)')
    plt.plot(ep, H.history['val_loss'], label='val_loss (1-DICE)')
    plt.title('U-Net - Loss'); plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(ep, H.history['dice_coef'], label='train DICE')
    plt.plot(ep, H.history['val_dice_coef'], label='val DICE')
    plt.title('U-Net - DICE'); plt.xlabel('Epoch'); plt.ylabel('DICE'); plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, 'curvas_unet.png'), dpi=120)
    plt.show()
else:
    print('U-Net cargada del disco -> no hay curvas de entrenamiento que mostrar.')
    print('  (Si quieres regenerarlas, borra modelo_unet.keras y vuelve a ejecutar.)')

## 6. Evaluacion en test (IoU / DICE)

Predecimos las mascaras del conjunto de test, las binarizamos en el umbral 0.5 y
calculamos IoU y DICE por imagen.

In [ ]:
pred = unet.predict(Xte, verbose=0)
pred_bin = (pred[..., 0] >= 0.5).astype('float32')

ious, dices = [], []
for i in range(len(Xte)):
    gt = Yte[i, ..., 0]; pr = pred_bin[i]
    inter = float((gt * pr).sum())
    union = float(((gt + pr) > 0).sum())
    ious.append(inter / (union + 1e-8))
    dices.append(2 * inter / (gt.sum() + pr.sum() + 1e-8))

unet_iou  = float(np.mean(ious))
unet_dice = float(np.mean(dices))
print('U-Net en test -> IoU medio = %.3f | DICE medio = %.3f' % (unet_iou, unet_dice))

# Guardamos resultados por imagen
pd.DataFrame({'IoU': ious, 'DICE': dices}).to_csv(
    os.path.join(OUT_DIR, 'unet_test_metrics.csv'), index=False)

In [ ]:
# Ejemplos visuales: imagen | mascara real | mascara predicha por U-Net
n_ej = 6
idx_ej = np.random.RandomState(SEED).choice(len(Xte), size=n_ej, replace=False)
plt.figure(figsize=(11, 14))
for fila, i in enumerate(idx_ej):
    plt.subplot(n_ej, 3, fila * 3 + 1)
    plt.imshow(Xte[i]); plt.axis('off'); plt.title('imagen')
    plt.subplot(n_ej, 3, fila * 3 + 2)
    plt.imshow(Xte[i]); plt.imshow(Yte[i, ..., 0], alpha=0.45, cmap='Greens')
    plt.axis('off'); plt.title('mascara real')
    plt.subplot(n_ej, 3, fila * 3 + 3)
    plt.imshow(Xte[i]); plt.imshow(pred_bin[i], alpha=0.45, cmap='Reds')
    plt.axis('off'); plt.title('U-Net pred (DICE=%.2f)' % dices[i])
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'unet_ejemplos.png'), dpi=120); plt.show()

## 7. Comparacion U-Net vs Grad-CAM del clasificador

Cargamos `localizacion.csv` (generado por el notebook 2) — contiene IoU/DICE del
**Grad-CAM** de VGG16 y ResNet50 sobre las imagenes con tumor del test — y lo
comparamos con los IoU/DICE de la **U-Net**, evaluados sobre **el mismo conjunto de
test**.

**Que esperar:** la U-Net es un segmentador real (entrenado con las mascaras), asi que
deberia tener IoU/DICE bastante mas altos que el Grad-CAM. El Grad-CAM es un
**subproducto** de un clasificador: te indica una **zona** donde mira la red, no una
mascara precisa. La comparacion cuantifica cuanto se queda el Grad-CAM por debajo de
una segmentacion propiamente entrenada.

In [ ]:
ruta_csv = os.path.join(OUT_DIR, 'localizacion.csv')
gradcam = pd.read_csv(ruta_csv)
res_gc = gradcam.groupby('modelo')[['IoU', 'DICE']].mean()

comp = res_gc.copy()
comp.loc['U-Net'] = [unet_iou, unet_dice]
comp.index.name = 'metodo'
print(comp.round(3))

comp.plot.bar(figsize=(8, 4), rot=0, color=['#4C72B0', '#55A868'])
plt.title('Localizacion del tumor: U-Net (segmentacion) vs Grad-CAM (clasificacion)')
plt.ylabel('valor medio'); plt.ylim(0, 1)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'comparacion_unet_gradcam.png'), dpi=120); plt.show()

## 8. Conclusiones

Rellena con tus resultados:

- **U-Net (segmentacion entrenada)**: IoU y DICE en test. Esperable mayor que el Grad-CAM.
- **Grad-CAM (clasificacion, byproduct)**: IoU/DICE mas bajos pero util para *explicar*
  donde mira el clasificador.
- **Interpretacion para el minipaper**: el Grad-CAM no sustituye a una segmentacion
  real, pero **confirma** que el clasificador atiende a la region del tumor (los
  numeros del Grad-CAM no son cero — el modelo sí mira el tumor, solo que de forma
  mas difusa que la U-Net).
- **Limitaciones**: pocas imagenes con tumor (~530), U-Net entrenada desde cero;
  ampliaciones posibles: data augmentation y BCE+DICE como loss.

Todos los resultados se guardan en `resultados/`.